# 07 · Profundidad y ResNet
*Por qué más capas entrenan peor, y la suma que lo arregla*

Notebook del episodio 7 de **Visión: de convoluciones a ViT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/vision/profundidad-y-resnet)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** familia CIFAR del paper (6n+2: 8, 20, 56 capas, con BN), 30 epochs SGD sin augmentation → residual cuesta +2.752 parámetros (2 proyecciones 1×1 de 27 bloques) → error train/test (2.000 imágenes): plana 0,6/19,8, 0,0/15,0, 1,1/26,9; residual 0,3/18,6, 0,0/15,8, 0,0/12,4 → curvas: en la epoch 10, plana-20 20 % vs. plana-56 47 % de error de train; residual-56 16 % → y = F(x) + x, F = 0 copia (|y − x| = 0) → grafo escalar: ∂y/∂x = 1 + 0,1 = 1,1 → norma del gradiente al inicio (56 capas): plana sin BN 0 en capa 1; plana con BN 140 vs. 0,6 (crece ~230× hacia la imagen); residual 2,7 vs. 1,7 → el paper dice que con BN no hay gradientes que se desvanecen (conjetura convergencia lenta) → paper: plana-34 28,54 % vs. plana-18 27,94 % top-1; ResNet-152 4,49 % top-5 sola, 3,57 % ensemble

**Cómo correrlo:** con el entorno de `~/Movies/serie-vision/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "vision"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Visión, de las convoluciones a los transformers. Episodio siete: profundidad y ResNet. Si una red de ocho capas anda bien, una de cincuenta y seis debería andar por lo menos igual. No pasa. Hoy lo medimos, entendemos por qué es raro, y escribimos la suma de una línea que permitió entrenar redes de cientos de capas.*

### La paradoja de la profundidad

- Más capas = más capacidad, en teoría
- En la práctica, a partir de cierta profundidad entrena peor
- No es overfitting: el error sube también en train
- ResNet: cada bloque aprende lo que falta, F(x), y suma x

Una red más profunda puede representar todo lo que representa una más chica, y más. Pero en la práctica, pasado cierto punto, agregar capas empeora el resultado. Y lo raro es dónde empeora: no solo en test, que sería overfitting, la red memorizando el train, sino en el propio error de entrenamiento. La solución de ResNet es que cada bloque no calcule la salida entera, sino una corrección F de x, que se suma a la entrada x.

### La red profunda podría copiar a la chica

El argumento de por qué es raro. Tomá la red de ocho capas ya entrenada. Agregale cuarenta y ocho capas que no hagan nada: que copien su entrada a la salida, la función identidad. Esa red de cincuenta y seis capas tiene exactamente el mismo error que la de ocho. Si el entrenamiento no encuentra esa solución, el problema no es de capacidad: el optimizador no llega. Y copiar la entrada con convoluciones y ReLU, rectified linear unit, no es tan fácil como parece.

imagen → 8 capas entrenadas → 48 capas identidad → misma salida

## Parte 1 · Medir la degradación

Seis redes en CIFAR-10, del Canadian Institute for Advanced Research: planas y residuales, de 8, 20 y 56 capas. Treinta epochs cada una, sin data augmentation.

### Cuántas capas y cuántos parámetros

Con n bloques por etapa, seis n más dos capas con pesos: ocho, veinte y cincuenta y seis. La plana apila convoluciones de tres por tres con BatchNorm y ReLU; la BatchNorm normaliza cada canal y la vemos en detalle en el episodio que viene. La residual tiene las mismas capas, más los atajos. La diferencia es de dos mil setecientos cincuenta y dos parámetros en todas: de los veintisiete bloques de la de cincuenta y seis, solo dos llevan una proyección de uno por uno, donde cambia la cantidad de canales. Los demás atajos son la identidad y no cuestan nada.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_parametros.py
__file__ = 'snippets/ep07_parametros.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _resnet import RedCifar, contar

In [3]:
print("capas   plana     residual   diferencia")
for n in [1, 3, 9]:                          # 6n + 2 capas con pesos
    plana = RedCifar(n=n, residual=False)
    resid = RedCifar(n=n, residual=True)
    p, r = contar(plana), contar(resid)
    print(f"{6 * n + 2:5} {p:9,} {r:10,} {r - p:+10,}")

bloques = resid.bloques                      # los 27 bloques de la de 56
proy = sum(not isinstance(b.atajo, torch.nn.Identity) for b in bloques)
print("bloques:", len(bloques), "| con proyección 1×1:", proy)

capas   plana     residual   diferencia
    8    75,290     78,042     +2,752
   20   269,722    272,474     +2,752
   56   853,018    855,770     +2,752
bloques: 27 | con proyección 1×1: 2


### Error de entrenamiento y de test

Cargamos las seis redes ya entrenadas. Medimos el error en diez mil imágenes de entrenamiento y en las diez mil de test. Primero las planas. Ocho capas: cero coma seis por ciento en train y diecinueve coma ocho en test. Veinte capas: cero en train y quince en test, mejor. Entre ocho y veinte, a esta escala, más capas ayudan. Pero cincuenta y seis capas: uno coma uno en train, y veintiséis coma nueve en test. La más profunda es la peor en las dos columnas. Las residuales: dieciocho coma seis, quince coma ocho y doce coma cuatro en test. Con atajos, la de cincuenta y seis es la mejor de las seis. Medido en dos mil imágenes de cada conjunto.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_resultados.py
__file__ = 'snippets/ep07_resultados.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _cifar import cargar, normalizar
from _resnet import cargar_red
from _redes import evaluar
torch.set_grad_enabled(False)
Xtr, ytr = cargar("train", n=2000); Xte, yte = cargar("test", n=2000)
Xtr, Xte = normalizar(Xtr), normalizar(Xte)

In [5]:
print("red          capas  error train  error test")
for nombre in ["plana8", "plana20", "plana56", "res8", "res20", "res56"]:
    red, info = cargar_red(f"{AQUI}/../datos/ep07_{nombre}.pt")
    e_tr = 1 - evaluar(red, Xtr, ytr)        # 2.000 imágenes de train
    e_te = 1 - evaluar(red, Xte, yte)        # 2.000 de test
    print(f"{nombre:12} {info['capas']:5}  {e_tr:10.1%}  {e_te:9.1%}")

red          capas  error train  error test


plana8           8        0.6%      19.8%


plana20         20        0.0%      15.0%


plana56         56        1.1%      26.9%


res8             8        0.3%      18.6%


res20           20        0.0%      15.8%


res56           56        0.0%      12.4%


### Error de train por epoch: redes planas

La historia completa del entrenamiento está en las curvas, y ahí la diferencia se ve mucho más que al final. La de ocho capas baja como esperaríamos. La de veinte baja parecido, y termina mejor. La de cincuenta y seis va muy por detrás todo el tiempo: en la epoch diez, la de veinte tiene veinte por ciento de error de entrenamiento y la de cincuenta y seis, cuarenta y siete. En la veinte, siete contra veintiuno. Al final las tres casi memorizan el train, pero la profunda llega tarde y generaliza peor. Eso es la degradación: no es que no pueda, es que el optimizador tarda mucho más.

> **He, Zhang, Ren y Sun, 2015 — Deep Residual Learning for Image Recognition** · [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)
>
> **plana 34 > plana 18** · «adding more layers to a suitably deep model leads to higher training error»; en ImageNet, error top-1: plana-18 27,94 %, plana-34 28,54 %.

El paper de ResNet lo describe así. Agregar capas a un modelo suficientemente profundo lleva a más error de entrenamiento. En ImageNet, mil clases y más de un millón de fotos, la red plana de treinta y cuatro capas tiene veintiocho coma cincuenta y cuatro por ciento de error top-1, peor que la de dieciocho, con veintisiete coma noventa y cuatro. Top-1 quiere decir que la primera respuesta de la red tiene que ser la correcta.

## Parte 2 · El bloque residual

Si copiar la entrada es difícil, hagamos que copiar sea lo que pasa por defecto.

### Aprender la diferencia

$$
y = F(x) + x
$$
<small>F: dos convoluciones; x: la entrada, sin tocar</small>

$$
F(x) = \mathrm{BN}(W_2\, \mathrm{relu}(\mathrm{BN}(W_1 x)))
$$
<small>W₁, W₂: convoluciones 3×3; BN: BatchNorm</small>

$$
F = 0 \;\Rightarrow\; y = x
$$
<small>copiar la entrada es lo más fácil: pesos en cero</small>

$$
y = F(x) + W_s\, x
$$
<small>si cambian canales o tamaño: Wₛ, convolución 1×1</small>

La salida del bloque es F de x más x. F son dos convoluciones de tres por tres, W uno y W dos, cada una seguida de BatchNorm, con una ReLU en el medio. La diferencia está en el caso fácil: si los pesos de F quedan en cero, el bloque devuelve x tal cual. Copiar la entrada pasa a ser el punto de partida. Cuando el bloque cambia la cantidad de canales, por ejemplo de dieciséis a treinta y dos, x no se puede sumar directo: se multiplica por W sub s, una convolución de uno por uno. En el dibujo, esa matriz mezclando canales en cada píxel.

### El bloque en PyTorch

El bloque en código. La rama F: convolución, BatchNorm, ReLU, convolución, BatchNorm. El atajo, por defecto, es la identidad. Si cambia el stride o la cantidad de canales, una convolución de uno por uno con el mismo stride. Y la salida suma las dos ramas y aplica la ReLU. Con dieciséis canales y stride uno, la forma no cambia. Con treinta y dos canales y stride dos, la salida tiene el doble de canales y la mitad de alto y de ancho, y el atajo se adapta.

**Cada línea, su término:**

- `f = self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))` → $F(x) = \mathrm{BN}(W_2\,\mathrm{relu}(\mathrm{BN}(W_1 x)))$ — la rama que aprende
- `self.atajo = nn.Identity()` → $x \mapsto x$ — el atajo: identidad, cero parámetros
- `self.atajo = nn.Conv2d(c_in, c_out, 1, stride, bias=False)` → $x \mapsto W_s x$ — proyección 1×1 si cambia la forma
- `return F.relu(f + self.atajo(x))` → $y = \mathrm{relu}(F(x) + x)$ — sumar y después la ReLU

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_bloque.py
__file__ = 'snippets/ep07_bloque.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

In [7]:
class BloqueResidual(nn.Module):
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.atajo = nn.Identity()               # el camino directo
        if stride != 1 or c_in != c_out:         # proyección 1×1
            self.atajo = nn.Conv2d(c_in, c_out, 1, stride, bias=False)

    def forward(self, x):
        f = self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))
        return F.relu(f + self.atajo(x))         # y = relu(F(x) + x)

x = torch.randn(8, 16, 32, 32)
print("igual:  ", tuple(BloqueResidual(16, 16)(x).shape))
print("achica: ", tuple(BloqueResidual(16, 32, stride=2)(x).shape))

igual:   (8, 16, 32, 32)
achica:  (8, 32, 16, 16)


### F = 0: la plana borra, la residual copia

Comprobamos el caso fácil. Un bloque plano y uno residual, iguales salvo el atajo. Ponemos en cero gamma, el peso de la última BatchNorm, así F de x da exactamente cero. La entrada es positiva, como la que sale de una ReLU. El bloque plano devuelve un tensor de norma cero: borró la imagen. El residual devuelve exactamente x: la diferencia con la entrada es cero, aunque la entrada tiene norma de casi cuarenta y cinco. En una red plana, aprender a no hacer nada requiere pesos muy precisos; en una residual, es lo que pasa con pesos en cero.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_identidad.py
__file__ = 'snippets/ep07_identidad.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _resnet import Bloque
torch.manual_seed(0)

In [9]:
plano = Bloque(16, 16, 1, residual=False, bn=True).eval()
resid = Bloque(16, 16, 1, residual=True, bn=True).eval()
for b in (plano, resid):
    torch.nn.init.zeros_(b.bn2.weight)   # γ = 0 en la última BN: F(x) = 0
x = torch.relu(torch.randn(4, 16, 8, 8))     # entrada ≥ 0, como tras una ReLU

with torch.no_grad():
    print("plano:    |y| =", round(plano(x).norm().item(), 2))
    print("residual: |y − x| =", round((resid(x) - x).norm().item(), 2),
          "  |x| =", round(x.norm().item(), 2))

plano:    |y| = 0.0
residual: |y − x| = 0.0   |x| = 44.95


### Un bloque residual con números

Un bloque de juguete con un solo número. Arriba, la rama F: multiplicar por w uno, que vale cero coma cinco, ReLU, y multiplicar por w dos, que vale cero coma dos. Abajo, el atajo: x va directo a la suma. Forward con x igual a dos: dos por cero coma cinco da uno, la ReLU lo deja en uno, por cero coma dos da cero coma dos, y la suma con x da dos coma dos. Backward, de derecha a izquierda: la derivada de y respecto de la suma es uno. Por la rama F, se multiplica por cero coma dos y por cero coma cinco: llega cero coma uno. Por el atajo llega uno entero. En x se suman: uno coma uno.

## Parte 3 · El camino del gradiente

La suma también cambia el backward.

### La derivada de una suma

$$
\frac{\partial y}{\partial x} = \frac{\partial F}{\partial x} + I
$$
<small>I: la matriz identidad, del atajo</small>

$$
\text{plana: } \textstyle\prod_{l=1}^{L} J_l \qquad \text{residual: } \prod_{l=1}^{L} (I + J_l)
$$
<small>regla de la cadena a través de L bloques; Jₗ = ∂Fₗ/∂x</small>

$$
J = 0{,}1,\ L = 5:\quad 0{,}1^5 = 0{,}00001 \quad\text{vs.}\quad 1{,}1^5 \approx 1{,}61
$$
<small>ejemplo con números, en una dimensión</small>

Derivamos y igual a F de x más x: la derivada de la suma es la suma de las derivadas, y la derivada de x respecto de sí mismo es la identidad I. Para llegar del final de la red a la primera capa, la regla de la cadena multiplica las derivadas de todos los bloques. En la plana, un producto de matrices J sub l. En la residual, un producto de identidad más J sub l: al expandirlo, uno de los términos es la identidad sola, el camino directo. Con números, en una dimensión: cinco bloques con derivada cero coma uno. La plana multiplica cero coma uno cinco veces y queda en diez a la menos cinco. La residual multiplica uno coma uno, y da uno coma seis.

### La norma del gradiente, capa por capa

Ahora medido, en redes de cincuenta y seis capas recién inicializadas. Un batch de doscientas cincuenta y seis imágenes reales y un backward. Tomamos las cincuenta y cinco convoluciones de tres por tres y la norma del gradiente de sus pesos. Tres variantes: plana sin BatchNorm, plana con BatchNorm, y residual con BatchNorm. Plana sin BatchNorm: el gradiente en la primera capa es cero, desapareció en el camino. Plana con BatchNorm: no desaparece, al revés, en la primera capa vale ciento cuarenta, contra cero coma seis en la última. Residual: dos coma siete y uno coma siete, parejo de punta a punta.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_gradientes.py
__file__ = 'snippets/ep07_gradientes.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
import torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _cifar import cargar, normalizar
from _resnet import RedCifar
X, y = cargar("train", n=256)
X = normalizar(X)

In [11]:
def norma_grad_por_capa(red):
    F.cross_entropy(red(X), y).backward()          # un batch de 256
    convs = [m for m in red.modules() if isinstance(m, torch.nn.Conv2d)
             and m.kernel_size == (3, 3)]
    return [c.weight.grad.norm().item() for c in convs]

perfil = {}
for nombre, res, bn in [("plana-56 sin BN", False, False),
                        ("plana-56 con BN", False, True),
                        ("residual-56 con BN", True, True)]:
    torch.manual_seed(0)
    g = norma_grad_por_capa(RedCifar(n=9, residual=res, bn=bn))
    print(f"{nombre:19} capa 1: {g[0]:.1e}  capa 55: {g[-1]:.1e}")
    perfil[nombre] = g

plana-56 sin BN     capa 1: 0.0e+00  capa 55: 1.2e-02


plana-56 con BN     capa 1: 1.4e+02  capa 55: 6.1e-01


residual-56 con BN  capa 1: 2.7e+00  capa 55: 1.7e+00


In [12]:
# Verificación extra (corrió junto con el video)
import json

### Gradiente por capa, 56 capas, al inicio

El perfil completo, en escala logarítmica: cada unidad hacia arriba es multiplicar por diez. La plana sin BatchNorm se hunde a medida que vamos hacia la imagen, hasta cero. La plana con BatchNorm hace lo contrario: crece de manera sostenida y termina en más de dos, cientos de veces más que al final. Con un solo learning rate, unas capas dan pasos enormes y otras casi no se mueven. La residual queda en una franja angosta en todas las capas.

### Lo que dice el paper, y lo que medimos

- El paper usa BatchNorm en las planas: no se desvanece
- Dice que la degradación no se explica por el gradiente
- Conjetura: la plana profunda converge exponencialmente lento
- Acá, al inicio: con BN, el gradiente crece 200 veces

El paper entrena las redes planas con BatchNorm, justamente para que la señal no se apague. Y dice que verificaron que los gradientes tienen normas sanas: según ellos, la degradación no se explica por gradientes que se desvanecen. Su hipótesis es que las redes planas profundas convergen con una velocidad exponencialmente baja. En nuestra medición, al inicio del entrenamiento, el gradiente con BatchNorm no se desvanece, pero cambia más de doscientas veces de una punta a la otra. Consistente con el paper, pero no es la explicación completa.

## Parte 4 · Planas contra residuales

Las mismas profundidades, con atajos.

### Error de train: 20 y 56 capas

Ahora las mismas curvas con atajos. La plana de veinte, punteada, y la residual de veinte, llena: en la epoch diez, once por ciento de error contra veinte. La plana de cincuenta y seis, punteada roja, la más lenta de todas. Y la residual de cincuenta y seis, verde: en la epoch diez tiene dieciséis por ciento, menos que la plana de veinte, y en la veinte, uno coma cuatro. Con atajos, la profundidad vuelve a ser una ventaja: la red más profunda entrena rápido y termina con el mejor error de test, doce coma cuatro por ciento.

> **He et al., 2015 — ResNet en ImageNet** · [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)
>
> **152 capas · 3,57 %** · Redes de «up to 152 layers—8x deeper than VGG nets»; un ensemble logra «3.57% error on the ImageNet test set» (top-5) y la ResNet-152 sola, 4,49 % top-5 en validación.

El resultado del paper. Con atajos entrenaron redes de hasta ciento cincuenta y dos capas, ocho veces más profundas que VGG, la red de referencia de ese momento. Una sola ResNet de ciento cincuenta y dos capas tiene cuatro coma cuarenta y nueve por ciento de error top-5 en validación: la respuesta correcta tiene que estar entre las cinco primeras. Y un ensemble de varias ResNet, tres coma cincuenta y siete en test, con el que ganaron la competencia de ImageNet de dos mil quince.

## Resumiendo

- Degradación: más capas, más error de train (no overfitting)
- Bloque residual: y = F(x) + x; con F = 0 copia la entrada
- Proyección 1×1 solo donde cambian canales: +2.752 parámetros
- Backward: ∏(I + J) tiene siempre el camino directo
- Medido: plana-56 26,9 % de error de test; residual-56 12,4 %
- Próximo: BatchNorm por dentro y data augmentation